# ThriftyChef — Offline Evaluation Notebook

This notebook reproduces the offline experimental pipeline for **ThriftyChef**, a context-aware hybrid recipe recommender designed to reduce household food waste on the Food.com corpus.

The workflow covers: data loading; training of popularity, content-based, collaborative (SVD), and hybrid models; ranking evaluation (**NDCG**, **MAP**, precision, recall, hit-rate, RMSE); waste-reduction simulation; cold-start analysis; context ablation; generative-AI comparison; and explainability via clustering and SHAP.

| Section | Content |
|---------|---------|
| 0–1 | Environment, data ingestion, exploratory analysis |
| 2 | Model definitions and training |
| 3 | Top-$N$ ranking metrics |
| 4 | Waste-reduction simulation |
| 5 | Cold-start and warm-up behaviour |
| 6 | Context-aware re-ranking ablation |
| 7 | Generative approaches (comparative) |
| 8 | Clustering and SHAP explainability |
| 9 | Discussion of findings |

**Repository:** https://github.com/NJDBSProjects20093736/FridgeWise

> **Reproducibility.** Experiment settings follow `CA_ONE_CONFIG` (seed `1103`). Processed tables are distributed as `data/presentation/data_clean.zip` and are unpacked automatically from the repository.


## 0. Environment setup

Install dependencies and resolve the project root. The notebook is intended for Google Colab and also runs locally when executed from the FridgeWise working tree.


In [ ]:
# Detect Colab vs local
import os, sys
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = "https://github.com/NJDBSProjects20093736/FridgeWise.git"
BRANCH = "Thrifty-chef-implementations"  # presentation branch with latest merge from main
PROJECT_DIR = Path("/content/FridgeWise") if IN_COLAB else (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)

if IN_COLAB:
    # Lightweight install for the demo (skip torch/faiss/API stack)
    %pip install -q pandas numpy scipy scikit-learn scikit-surprise matplotlib seaborn tqdm shap python-dateutil

    if not (PROJECT_DIR / "src").exists():
        !git clone --branch {BRANCH} --single-branch {REPO_URL} {PROJECT_DIR}
    else:
        print("Repo already present:", PROJECT_DIR)

    %cd {PROJECT_DIR}
else:
    print("Local mode — using:", PROJECT_DIR)

ROOT = PROJECT_DIR.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print("IN_COLAB =", IN_COLAB)
print("ROOT     =", ROOT)
print("src OK   =", (ROOT / "src").exists())


## 0.1 Dataset acquisition

Processed evaluation tables are versioned in the repository archive `data/presentation/data_clean.zip`. The following cell extracts them into `data/clean/`, or downloads the archive from GitHub if it is absent from the working copy.


In [ ]:
# Fetch clean CSVs from GitHub (bundled zip in the repo)
from pathlib import Path
import zipfile, shutil, urllib.request

CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)
ZIP_IN_REPO = ROOT / "data" / "presentation" / "data_clean.zip"

# Same file on GitHub (used if the clone somehow lacks the zip)
DATA_BRANCH = "Thrifty-chef-implementations"
GITHUB_ZIP_URL = (
    "https://raw.githubusercontent.com/NJDBSProjects20093736/FridgeWise/"
    f"{DATA_BRANCH}/data/presentation/data_clean.zip"
)

def unzip_clean(zip_path: Path) -> None:
    extract_root = ROOT / "data" / "_unzip_clean"
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_root)
    hits = list(extract_root.rglob("clean_recipes.csv"))
    if not hits:
        raise FileNotFoundError("clean_recipes.csv not found inside data_clean.zip")
    for f in hits[0].parent.glob("*"):
        if f.is_file():
            shutil.copy2(f, CLEAN / f.name)

if (CLEAN / "clean_recipes.csv").exists():
    print("data/clean already present — skipping download")
elif ZIP_IN_REPO.exists():
    print("Using zip from cloned repo:", ZIP_IN_REPO)
    unzip_clean(ZIP_IN_REPO)
else:
    print("Zip missing locally — downloading from GitHub...")
    print(GITHUB_ZIP_URL)
    dest = ROOT / "data" / "presentation"
    dest.mkdir(parents=True, exist_ok=True)
    out = dest / "data_clean.zip"
    with urllib.request.urlopen(GITHUB_ZIP_URL, timeout=180) as resp:
        out.write_bytes(resp.read())
    print(f"Downloaded {out.stat().st_size / 1e6:.1f} MB")
    unzip_clean(out)

print("clean_recipes.csv present:", (CLEAN / "clean_recipes.csv").exists())
print("Files in data/clean:", sorted(p.name for p in CLEAN.glob("*"))[:20])


In [ ]:
# Sanity check required tables
required = [
    "clean_recipes.csv",
    "clean_interactions.csv",
    "user_fridge_inventory.csv",
    "user_profiles.csv",
    "context_tag_lifts.csv",
    "recipe_ingredient_features.csv",
]
missing = [name for name in required if not (ROOT / "data" / "clean" / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing clean tables: "
        + ", ".join(missing)
        + " — push data/presentation/data_clean.zip to GitHub and re-run setup."
    )
print("All required clean tables found.")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiment import CA_ONE_CONFIG

RANDOM_STATE = CA_ONE_CONFIG.random_state  # 1103
K = CA_ONE_CONFIG.k                        # 10
MAX_EVAL_USERS = CA_ONE_CONFIG.max_eval_users  # 500 — lower for a faster live run
FAST_DEMO = IN_COLAB  # quicker charts during presentation
if FAST_DEMO:
    MAX_EVAL_USERS = min(MAX_EVAL_USERS, 200)

np.random.seed(RANDOM_STATE)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(f"Seed={RANDOM_STATE} | K={K} | eval_users={MAX_EVAL_USERS} | FAST_DEMO={FAST_DEMO}")


In [ ]:
from src.data_loader import load_fridgewise_data, parse_json_list

data = load_fridgewise_data(ROOT)

n_users = data.interactions["user_id"].nunique()
n_items = data.interactions["recipe_id"].nunique()
print(f"Recipes:      {len(data.recipes):>8,}")
print(f"Interactions: {len(data.interactions):>8,}")
print(f"Users:        {n_users:>8,}")
print(f"Fridge items: {len(data.fridge):>8,}  |  User profiles: {len(data.profiles):>8,}")
display(data.recipes[["recipe_id", "recipe_name", "minutes", "n_ingredients"]].head(3))


### 1.1 Exploratory data analysis

Food.com interactions are heavily skewed toward high ratings (predominantly 4–5). Under such skew, pointwise error metrics such as RMSE can appear optimistic. Consequently, the primary offline protocol emphasises ranking measures (NDCG and MAP), which evaluate the quality of the recommended list relative to held-out relevant items.


In [ ]:
dist = (data.interactions["rating"].value_counts(normalize=True).sort_index() * 100)
sparsity = 1 - len(data.interactions) / (n_users * n_items)
print("Rating distribution (%):")
print(dist.round(1).to_string())
print(f"\nSparsity: {sparsity:.4%}  |  density: {1-sparsity:.4%}")

fig, ax = plt.subplots(figsize=(6, 3.5))
dist.plot(kind="bar", ax=ax, color="#2980b9")
ax.set(title="Rating distribution", xlabel="rating", ylabel="% of interactions")
plt.tight_layout(); plt.show()


## 2. Recommendation models

Four recommenders of increasing complexity are trained on the same per-user hold-out split:

| Model | Description |
|-------|-------------|
| Popularity | Global baseline using Bayesian-smoothed interaction frequency and mean rating |
| Content-based | TF–IDF over ingredients and tags, combined with fridge-ingredient overlap |
| Collaborative (SVD) | Matrix factorisation of the user–recipe rating matrix |
| Hybrid | Weighted fusion of collaborative scores with fridge match, expiry priority, nutrition, and optional contextual boosts |

The popularity model provides a strong baseline on sparse catalogues. The hybrid model is the candidate production configuration because it jointly optimises relevance and inventory utilisation.


In [ ]:
from src.evaluation.splits import user_holdout_split
from src.models.popularity import PopularityRecommender
from src.models.content_based import ContentBasedRecommender
from src.models.collaborative import CollaborativeRecommender
from src.recommender import HybridRecommender

train_df, test_df, test_relevant = user_holdout_split(
    data.interactions, random_state=RANDOM_STATE
)
train_data = data.with_interactions(train_df)
print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | users with relevant hold-out: {len(test_relevant):,}")

popularity = PopularityRecommender().fit(train_data)
content = ContentBasedRecommender().fit(train_data)
svd = CollaborativeRecommender(n_factors=50, n_epochs=20).fit(
    train_data, test_size=0.0, random_state=RANDOM_STATE
)
hybrid = HybridRecommender().fit(train_data, svd, content)
hybrid_no_ctx = HybridRecommender(context_max_boost=0.0).fit(train_data, svd, content)
print("Trained: popularity, content_based, svd, hybrid (+ no-context ablation)")


## 3. Offline ranking evaluation (NDCG and MAP)

For each evaluation user, each model returns a top-$K$ list of unseen recipes. Relevance is defined by held-out interactions with rating $\geq 4$. We report precision, recall, mean average precision (MAP), normalised discounted cumulative gain (NDCG), and hit-rate at $K$, averaged over a fixed random sample of users (seeded for reproducibility).


In [ ]:
from src.evaluation.metrics import (
    precision_at_k, recall_at_k, average_precision_at_k, ndcg_at_k, hit_rate_at_k,
)
from src.evaluation.evaluator import evaluate_svd_rmse

_rng = np.random.default_rng(RANDOM_STATE)
eval_users = list(test_relevant.keys())
if len(eval_users) > MAX_EVAL_USERS:
    eval_users = list(_rng.choice(eval_users, MAX_EVAL_USERS, replace=False))

def evaluate(model, name):
    acc = {"precision": [], "recall": [], "map": [], "ndcg": [], "hit_rate": []}
    for uid in eval_users:
        rel = test_relevant[uid]
        recs = [r.recipe_id for r in model.recommend(int(uid), k=K, exclude_seen=True)]
        acc["precision"].append(precision_at_k(recs, rel, K))
        acc["recall"].append(recall_at_k(recs, rel, K))
        acc["map"].append(average_precision_at_k(recs, rel, K))
        acc["ndcg"].append(ndcg_at_k(recs, rel, K))
        acc["hit_rate"].append(hit_rate_at_k(recs, rel, K))
    return {"model": name, **{m: float(np.mean(v)) for m, v in acc.items()}}

ranking = pd.DataFrame([
    evaluate(popularity, "popularity"),
    evaluate(content, "content_based"),
    evaluate(svd, "svd"),
    evaluate(hybrid, "hybrid"),
]).set_index("model")

rmse = evaluate_svd_rmse(data.interactions, random_state=RANDOM_STATE)
print(f"SVD RMSE (80/20): {rmse:.4f}")
display(ranking)

ax = ranking[["ndcg", "map", "hit_rate"]].plot(
    kind="bar", figsize=(8, 4), color=["#2980b9", "#27ae60", "#e67e22"]
)
ax.set(title=f"Ranking metrics @{K}", ylabel="score")
plt.xticks(rotation=15); plt.legend(loc="upper right"); plt.tight_layout(); plt.show()


## 4. Waste-reduction simulation

Beyond ranking relevance, the system objective includes reducing food waste. The simulation measures **waste coverage**: the fraction of soon-to-expire fridge items (`days_to_expiry` $\leq 5$) that appear among ingredients of the top-$K$ recommended recipes. Plotting NDCG against waste coverage exposes the relevance–sustainability trade-off across models.


In [ ]:
from src.evaluation.waste import simulate_waste_reduction

waste = pd.DataFrame([
    {
        "model": w.model_name,
        "waste_coverage": w.waste_coverage,
        "expiring_used": w.expiring_items_used,
        "expiring_total": w.expiring_items_total,
    }
    for w in (simulate_waste_reduction(m, data, k=K) for m in [popularity, content, hybrid])
]).set_index("model")
display(waste)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for m in ["popularity", "content_based", "hybrid"]:
    nd, wc = ranking.loc[m, "ndcg"], waste.loc[m, "waste_coverage"]
    ax.scatter(nd, wc, s=160)
    ax.annotate(m, (nd, wc), textcoords="offset points", xytext=(8, 6))
ax.set(
    xlabel=f"NDCG@{K} (relevance)",
    ylabel="Waste coverage (expiring items used)",
    title="Relevance vs waste-reduction trade-off",
)
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 5. Cold-start mitigation

Users without rating history cannot benefit from collaborative filtering. ThriftyChef therefore falls back to content and inventory signals (fridge match and expiry). The warm-up curve reports ingredient-match quality as the number of observed ratings increases, illustrating the transition from cold-start heuristics to personalised ranking.


In [ ]:
from src.evaluation.cold_start import (
    evaluate_new_user_fallback, evaluate_warmup_curve, evaluate_substitutions,
)

cold = pd.DataFrame([
    {"model": c.model, "new_user_match": c.mean_ingredient_match, "waste": c.waste_coverage}
    for c in evaluate_new_user_fallback(data, hybrid, content, popularity, k=K)
]).set_index("model")
display(cold)

warm = pd.DataFrame(evaluate_warmup_curve(hybrid, data, k=K))
display(warm[["num_ratings", "users_evaluated", "mean_ingredient_match", "waste_coverage", "cold_start_mode"]])

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(warm["num_ratings"], warm["mean_ingredient_match"], "o-", color="#2980b9")
ax1.set(xlabel="ratings in history", ylabel="ingredient match @10",
        title="Cold-start warm-up (hybrid)")
ax1.set_xticks(warm["num_ratings"]); plt.tight_layout(); plt.show()

alias_path = ROOT / "assets" / "ingredient_aliases.csv"
if alias_path.exists():
    subs = evaluate_substitutions(data, alias_path)
    for ex in subs[:3]:
        print(ex["recipe_name"])
        for s in ex["substitutions"][:2]:
            print(f"    missing '{s['missing']}'  ->  use '{s['substitute']}'  ({s['reason']})")
else:
    print("Alias file missing — skip substitution examples")


## 6. Context-aware re-ranking (CARS)

The hybrid model optionally applies small season- and weekday-based tag-lift adjustments (context-aware re-ranking). An ablation compares hybrid ranking with and without these boosts. Empirically, contextual re-ranking did not improve NDCG on this dataset; the serving default therefore disables context (`use_context=False`) unless explicitly enabled.


In [ ]:
ctx_on = evaluate(hybrid, "hybrid_with_context")
ctx_off = evaluate(hybrid_no_ctx, "hybrid_without_context")
print(f"NDCG@{K} with context:    {ctx_on['ndcg']:.4f}")
print(f"NDCG@{K} without context: {ctx_off['ndcg']:.4f}")
print(f"Delta: {ctx_on['ndcg'] - ctx_off['ndcg']:+.4f}")
pd.DataFrame([ctx_on, ctx_off]).set_index("model")[["ndcg", "map", "hit_rate"]]


## 7. Generative AI: comparative analysis

Generative methods (e.g. Mult-VAE, GANs, and large language models) are assessed as complementary components rather than replacements for the classical hybrid stack. Safety-critical constraints—allergen and dietary hard filters—remain deterministic. Generative models are better suited to optional ranking variants or natural-language explanations of recommendations.


In [ ]:
from src.genai.analysis import (
    genai_comparison_table, recommend_genai_strategy, sample_llm_prompt,
)

genai_df = pd.DataFrame([
    {
        "approach": g.name,
        "type": g.type,
        "implemented": g.implemented,
        "role": g.fit_for_fridgewise,
        "pros": "; ".join(g.pros[:2]),
        "cons": "; ".join(g.cons[:2]),
    }
    for g in genai_comparison_table()
])
display(genai_df)

print("Recommended GenAI strategy:")
for key, val in recommend_genai_strategy().items():
    print(f"  - {key}: {val}")

print("\nExample LLM explanation prompt:")
print(" ", sample_llm_prompt(
    "Greek Potatoes Oven Roasted", 0.5, ["parsley", "garlic powder"]
))


## 8. Lecture extensions: clustering and SHAP

Two interpretability and cold-start extensions are demonstrated. Socio-demographic **K-means** clustering of user profiles supports persona-level fallback. A **SHAP** explanation over a logistic surrogate identifies the feature contributions (ingredient match, expiry priority, predicted rating, nutrition, cook time, recipe size) that drive an individual recommendation.


In [ ]:
# Clustering (profiles)
from src.models.cold_start_clustering import (
    evaluate_cluster_model, compute_wcss_curve,
)
from src.models.case_based import build_fridge_cases, CaseBasedRecommender
from src.models.explainability import RecommendationExplainer

metrics = evaluate_cluster_model(data.profiles, n_clusters=5)
print(f"K-Means profiles — silhouette={metrics.silhouette:.4f}, davies_bouldin={metrics.davies_bouldin:.4f}")
display(compute_wcss_curve(data.profiles).head())

cases = build_fridge_cases(data)
cbr = CaseBasedRecommender(cases)
print(f"Case-based fridge cases: {len(cases)}")
if not cases.empty:
    sample = cases.iloc[0]
    raw_ings = sample["fridge_ingredients"]
    ings = parse_json_list(raw_ings) if isinstance(raw_ings, str) else list(raw_ings)
    explained = cbr.explain_recommendation(
        ings,
        sample.get("dietary_preference", "none"),
        sample.get("preferred_cuisine", "any"),
    )
    print(explained.get("text") if isinstance(explained, dict) else explained)


In [ ]:
# SHAP surrogate explanation on a sample user-recipe pair
explainer = RecommendationExplainer(random_state=RANDOM_STATE).fit(data)
print("SHAP surrogate fitted:", explainer._fitted)

feat_table = explainer.build_feature_table(data, sample_size=1500)
if explainer._fitted and not feat_table.empty:
    row = feat_table.iloc[0]
    result = explainer.explain_instance(row, int(row["user_id"]), int(row["recipe_id"]))
    for line in explainer.explanation_bullets(result):
        print(line)
    if result is not None:
        names = [f["feature"] for f in result.top_features]
        vals = [f["shap_value"] for f in result.top_features]
        fig, ax = plt.subplots(figsize=(7, 3.5))
        colors = ["#27ae60" if v > 0 else "#c0392b" for v in vals]
        ax.barh(names[::-1], vals[::-1], color=colors[::-1])
        ax.set(
            title=f"SHAP drivers — user {result.user_id} / recipe {result.recipe_id}",
            xlabel="SHAP value",
        )
        ax.axvline(0, color="black", lw=0.8)
        plt.tight_layout(); plt.show()
else:
    print("SHAP not available for this sample — check profiles/fridge coverage.")


## 9. Discussion and conclusions

The following cell aggregates the principal quantitative outcomes for reporting and discussion.


In [ ]:
summary = {
    "seed": RANDOM_STATE,
    "k": K,
    "eval_users": len(eval_users),
    "svd_rmse": round(float(rmse), 4),
    "ndcg": ranking["ndcg"].round(4).to_dict(),
    "map": ranking["map"].round(4).to_dict(),
    "waste_coverage": waste["waste_coverage"].round(4).to_dict(),
    "cold_start_match": cold["new_user_match"].round(4).to_dict(),
    "context_delta_ndcg": round(ctx_on["ndcg"] - ctx_off["ndcg"], 4),
    "shap_fitted": bool(explainer._fitted),
}

print("=== ThriftyChef offline evaluation summary ===")
for key, val in summary.items():
    print(f"{key}: {val}")

print("""
Principal findings
1. Rating prediction (SVD RMSE) is strong, but the rating distribution is heavily skewed toward 4-5,
   so ranking metrics are the more informative protocol.
2. On top-N relevance (NDCG/MAP), the popularity baseline remains competitive under catalogue sparsity
   and popularity bias; among personalised models, the hybrid closes most of the gap to popularity.
3. On the application objective, fridge-aware models (hybrid and content-based) achieve substantially
   higher waste coverage and stronger cold-start ingredient match than popularity alone.
4. Contextual tag-lift re-ranking yields a negligible or negative change in NDCG on this split;
   therefore context is disabled by default at serving time.
5. Generative methods are positioned as complementary (explanations / optional latent models);
   allergen and diet constraints remain hard filters.
6. SHAP attributions provide instance-level explanations of hybrid feature contributions,
   supporting transparency requirements for the recommender.
""")


## Execution notes (Google Colab)

1. Ensure the branch containing `data/presentation/data_clean.zip` is available on GitHub.
2. Open this notebook from the repository in Colab.
3. Execute all cells in order; the dataset archive is retrieved from GitHub automatically.
